# Correlação Intra Classe para verificar consistência entre as medidas.

### Bibliotecas utilizadas

In [10]:
import pandas as pd
import pingouin as pg

### Lendo a base

In [11]:
caminho = r'C:\Users\saulosgil\OneDrive\Documentos\functional_tests_blinding\data\data_blinding.xlsx'
df = pd.read_excel(caminho)

df.head()

,id,idade,estado_civil,sexo,raca,renda,baixa_visao,cego,comorbidades,medicamentos,peso_kg,estatura_cm,imc,tug_1,tug_2,ts_1,ts_2
0,5,19,Solteiro,Feminino,Branco,Até 1 salário mínimo,1,0,Pre diabete,Loratadina e clenil,55.0,158,22.031726,8.65,6.54,15,18
1,9,18,Solteiro,Feminino,Preto,Até 1 salário mínimo,1,0,Ansiedade,Nenhum,70.0,171,23.938990,6.09,5.51,11,13
2,10,62,Divorciado,Masculino,Branco,1 – 2 salários mínimos,1,0,Nenhuma comorbidade,Nenhum nenhuma,85.0,180,26.234568,7.89,9.77,14,15
3,11,42,Solteiro,Masculino,Preto,1 – 2 salários mínimos,1,0,Nenhuma comorbidade,Não,105.0,168,37.202381,12.80,8.75,12,15
4,17,42,Solteiro,Masculino,Preto,Até 1 salário mínimo,1,0,Nenhuma comorbidade,Não,75.0,165,27.548209,10.00,8.10,10,12


## **Selecionar apenas os cegos**

In [12]:
df = df[df['cego'] == 1]
df.head()

,id,idade,estado_civil,sexo,raca,renda,baixa_visao,cego,comorbidades,medicamentos,peso_kg,estatura_cm,imc,tug_1,tug_2,ts_1,ts_2
21,1,55,Solteiro,Masculino,Preto,Mais de 4 salários mínimos,0,1,Nenhuma comorbidade,Nenhum,99.0,172,33.464035,6.75,5.13,11,16
22,2,32,Solteiro,Masculino,Preto,Mais de 4 salários mínimos,0,1,Nenhuma comorbidade,Nenhum,92.0,183,27.471707,9.72,9.70,15,18
23,4,30,Solteiro,Masculino,Branco,Até 1 salário mínimo,0,1,Diabetes,"Sim, insulina",60.0,165,22.038567,14.00,12.30,10,12
24,6,20,Solteiro,Masculino,Branco,Mais de 4 salários mínimos,0,1,Depressão,Faço \n,60.0,180,18.518519,6.93,5.12,11,15
25,7,22,Solteiro,Masculino,Branco,1 – 2 salários mínimos,0,1,Nenhuma comorbidade,Nenhum,57.0,168,20.195578,6.75,5.13,11,16


### Média, SD e difference (session 2 - session 1) - TUG

In [13]:
mean1 = df['tug_1'].mean()
stfd1 = df['tug_1'].std()
print(f"Media: {mean1}; SD {stfd1};")

mean2 = df['tug_2'].mean()
stfd2 = df['tug_2'].std()
print(f"Media: {mean2}; SD {stfd2};")

diff_mean = (df['tug_2'] - df['tug_1']).mean()
diff_std = (df['tug_2'] - df['tug_1']).std()
print(f"Diff: {diff_mean}; SD {diff_std};")


Media: 10.568888888888889; SD 4.128798794707668;
Media: 9.277037037037037; SD 4.1420561735362;
Diff: -1.2918518518518516; SD 1.7507120244610739;


### Média, SD e difference (session 2 - session 1) - TS

In [14]:
mean1 = df['ts_1'].mean()
stfd1 = df['ts_1'].std()
print(f"Media: {mean1}; SD {stfd1};")

mean2 = df['ts_2'].mean()
stfd2 = df['ts_2'].std()
print(f"Media: {mean2}; SD {stfd2};")

diff_mean = (df['ts_2'] - df['ts_1']).mean()
diff_std = (df['ts_2'] - df['ts_1']).std()
print(f"Diff: {diff_mean}; SD {diff_std};")

Media: 10.703703703703704; SD 2.643057885434026;
Media: 13.518518518518519; SD 3.017517705898858;
Diff: 2.814814814814815; SD 1.3311948663061215;


## **Time Up and Go**

### Organizando os dados do TUG - wide to long

In [15]:
# Selecionar as colunas necessárias
tug_icc = df[['id', 'tug_1', 'tug_2']].copy()

# Transformar do formato wide para long
tug_long = pd.melt(
    tug_icc,
    id_vars='id',
    value_vars=['tug_1', 'tug_2'],
    var_name='day',
    value_name='score'
)

# Organizar a ordem
tug_long['day'] = tug_long['day'].map({
    'tug_1': 1,
    'tug_2': 2
})

tug_long = tug_long.sort_values(['id', 'day']).reset_index(drop=True)

tug_long.head(5)

,id,day,score
0,1,1,6.75
1,1,2,5.13
2,2,1,9.72
3,2,2,9.70
4,4,1,14.00


### Cálculo do ICC (modelo de dois fatores, consistência, medidas médias)

In [16]:
icc_result = pg.intraclass_corr(data=tug_long,
                                targets='id',
                                raters='day',
                                ratings='score')

# print(icc_result) - ICC3 consistência entre medidas
icc_3 = icc_result.set_index('Type').loc[['ICC(C,1)']]
icc_3



,ICC,F,df1,df2,pval,CI95
Type,,,,,,
"ICC(C,1)",0.91039,21.318885,26,26,9.164388e-12,"[0.81, 0.96]"


## **Timed Stands**

### Organizando os dados do TS - wide to long

In [17]:
# Selecionar as colunas necessárias
ts_icc = df[['id', 'ts_1', 'ts_2']].copy()

# Transformar do formato wide para long
ts_long = pd.melt(
    ts_icc,
    id_vars='id',
    value_vars=['ts_1', 'ts_2'],
    var_name='day',
    value_name='score'
)

# Organizar a ordem
ts_long['day'] = ts_long['day'].map({
    'ts_1': 1,
    'ts_2': 2
})

ts_long = ts_long.sort_values(['id', 'day']).reset_index(drop=True)

ts_long.head(5)

,id,day,score
0,1,1,11
1,1,2,16
2,2,1,15
3,2,2,18
4,4,1,10


### Cálculo do ICC (modelo de dois fatores, consistência, medidas médias)

In [18]:
icc_result = pg.intraclass_corr(data=ts_long,
                                targets='id',
                                raters='day',
                                ratings='score')

# print(icc_result) - ICC3 consistência entre medidas
icc_3 = icc_result.set_index('Type').loc[['ICC(C,1)']]
icc_3


,ICC,F,df1,df2,pval,CI95
Type,,,,,,
"ICC(C,1)",0.889873,17.160772,26,26,1.186303e-10,"[0.77, 0.95]"
